In [2]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
from google.colab import files

uploaded = files.upload()

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

print("Movies dataset:", movies.shape)
print("Credits dataset:", credits.shape)

Saving tmdb_5000_credits.csv to tmdb_5000_credits.csv
Saving tmdb_5000_movies.csv to tmdb_5000_movies.csv
Movies dataset: (4803, 20)
Credits dataset: (4803, 4)


In [4]:
print("Movie Dataset: ")
display(movies.head())

print("Columns: ")
print(movies.columns.tolist())

Movie Dataset: 


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


Columns: 
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [5]:
print("Data types: ")
print(movies.dtypes)

print("\nMissing values: ")
print(movies.isnull().sum().sort_values(ascending=False).head(10))

Data types: 
budget                    int64
genres                   object
homepage                 object
id                        int64
keywords                 object
original_language        object
original_title           object
overview                 object
popularity              float64
production_companies     object
production_countries     object
release_date             object
revenue                   int64
runtime                 float64
spoken_languages         object
status                   object
tagline                  object
title                    object
vote_average            float64
vote_count                int64
dtype: object

Missing values: 
homepage          3091
tagline            844
overview             3
runtime              2
release_date         1
id                   0
budget               0
genres               0
original_title       0
popularity           0
dtype: int64


In [6]:
print("Duplicate movie rows: ",movies.duplicated().sum())
print("Duplicate movie titles: ",movies["title"].duplicated().sum())

Duplicate movie rows:  0
Duplicate movie titles:  3


In [8]:
movies=movies[["id","title","genres","keywords","overview"]].copy()
print("Selected columns: ")
print(movies.columns.tolist())
display(movies.head(3))

Selected columns: 
['id', 'title', 'genres', 'keywords', 'overview']


,id,title,genres,keywords,overview
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",A cryptic message from Bond’s past sends him o...


In [9]:
print("Missing values before cleaning:")
print(movies[["genres", "keywords", "overview"]].isnull().sum())

movies["genres"] = movies["genres"].fillna("[]")
movies["keywords"] = movies["keywords"].fillna("[]")
movies["overview"] = movies["overview"].fillna("")

print("\nMissing values after cleaning:")
print(movies[["genres", "keywords", "overview"]].isnull().sum())

Missing values before cleaning:
genres      0
keywords    0
overview    3
dtype: int64

Missing values after cleaning:
genres      0
keywords    0
overview    0
dtype: int64


In [10]:
def extract_names(text):
    items = ast.literal_eval(text)
    return " ".join(item["name"] for item in items)

movies["genres"] = movies["genres"].apply(extract_names)
movies["keywords"] = movies["keywords"].apply(extract_names)

display(movies[["title", "genres", "keywords"]].head())

,title,genres,keywords
0,Avatar,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...
2,Spectre,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...
3,The Dark Knight Rises,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...
4,John Carter,Action Adventure Science Fiction,based on novel mars medallion space travel pri...


In [11]:
movies["content"] = (
    movies["genres"] + " " +
    movies["keywords"] + " " +
    movies["overview"]
)

display(movies[["title", "content"]].head(3))

,title,content
0,Avatar,Action Adventure Fantasy Science Fiction cultu...
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action ocean drug abuse exot...
2,Spectre,Action Adventure Crime spy based on novel secr...


In [12]:
movie_index=0
print("Movie: ",movies.iloc[movie_index]["title"])
print("\n Movie content:")
print(movies.iloc[movie_index]["content"])

Movie:  Avatar

 Movie content:
Action Adventure Fantasy Science Fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.


In [13]:
sample_text=movies.iloc[0]["content"]
tokens=sample_text.lower().split()
print("Orignal text: ")
print(sample_text[:200])
print("\nFirst 20 tokens: ")
print(tokens[:20])

Orignal text: 
Action Adventure Fantasy Science Fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti w

First 20 tokens: 
['action', 'adventure', 'fantasy', 'science', 'fiction', 'culture', 'clash', 'future', 'space', 'war', 'space', 'colony', 'society', 'space', 'travel', 'futuristic', 'romance', 'space', 'alien', 'tribe']


In [14]:
from sklearn.feature_extraction.text import CountVectorizer

sample_movies = [
    "space adventure space",
    "space adventure",
    "crime adventure"
]

bow = CountVectorizer()
bow_matrix = bow.fit_transform(sample_movies)

print("Vocabulary:", bow.get_feature_names_out())
print("\nBag of Words matrix:")
print(bow_matrix.toarray())

Vocabulary: ['adventure' 'crime' 'space']

Bag of Words matrix:
[[1 0 2]
 [1 0 1]
 [1 1 0]]


In [15]:
sample_tfidf = TfidfVectorizer()
sample_matrix = sample_tfidf.fit_transform(sample_movies)

print("Vocabulary:", sample_tfidf.get_feature_names_out())
print("\nTF-IDF matrix:")
print(np.round(sample_matrix.toarray(), 2))

Vocabulary: ['adventure' 'crime' 'space']

TF-IDF matrix:
[[0.36 0.   0.93]
 [0.61 0.   0.79]
 [0.51 0.86 0.  ]]


In [16]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(movies["content"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (4803, 5000)
Vocabulary size: 5000


In [17]:
feature_names = tfidf.get_feature_names_out()

movie_vector = tfidf_matrix[0]

print("Movie:", movies.iloc[0]["title"])
print("Vector shape:", movie_vector.shape)
print("Non-zero features:", movie_vector.nnz)

print("\nFirst 10 vocabulary words:")
print(feature_names[:10])

Movie: Avatar
Vector shape: (1, 5000)
Non-zero features: 40

First 10 vocabulary words:
['000' '007' '10' '100' '11' '12' '13' '14' '15' '15th']


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_example = cosine_similarity(
    sample_matrix[0:1],
    sample_matrix
)

print("Similarity scores:")
print(np.round(similarity_example, 2))

Similarity scores:
[[1.   0.96 0.18]]


In [19]:
cosine_sim = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", cosine_sim.shape)

print("\nFirst 5 movie titles:")
print(movies["title"].head().tolist())

print("\nSimilarity scores for the first movie:")
print(np.round(cosine_sim[0][:5], 3))

Similarity matrix shape: (4803, 4803)

First 5 movie titles:
['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter']

Similarity scores for the first movie:
[1.    0.025 0.01  0.015 0.229]


In [20]:
indices = pd.Series(
    movies.index,
    index=movies["title"].str.lower()
).drop_duplicates()

print("Total unique movie titles:", len(indices))
print("\nExample lookup:")
print("Inception index:", indices.get("inception", "Not found"))

Total unique movie titles: 4803

Example lookup:
Inception index: 96


In [21]:
def recommend_movies(title, top_n=5):
    title = title.lower().strip()

    if title not in indices:
        return f"Movie '{title}' not found. Please check the title."

    movie_idx = indices[title]
    scores = list(enumerate(cosine_sim[movie_idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [item for item in scores if item[0] != movie_idx]

    top_movies = scores[:top_n]
    movie_indices = [item[0] for item in top_movies]

    return movies.iloc[movie_indices][["title", "genres"]].reset_index(drop=True)

In [22]:
recommend_movies("Inception")

,title,genres
0,Duplex,Action Comedy Thriller
1,Cypher,Thriller Science Fiction Mystery
2,The Helix... Loaded,Action Comedy Science Fiction
3,The Count of Monte Cristo,Action Adventure Drama Thriller
4,Pitch Perfect 2,Comedy Music


In [23]:
print("Recommendations for The Dark Knight:")
display(recommend_movies("The Dark Knight"))

print("\nTesting an invalid title:")
print(recommend_movies("Random Movie XYZ"))

Recommendations for The Dark Knight:


,title,genres
0,The Dark Knight Rises,Action Crime Drama Thriller
1,Batman Forever,Action Crime Fantasy
2,Batman Returns,Action Fantasy
3,Batman Begins,Action Crime Drama
4,Batman v Superman: Dawn of Justice,Action Adventure Fantasy



Testing an invalid title:
Movie 'random movie xyz' not found. Please check the title.


In [24]:
test_movies = ["Interstellar", "Toy Story", "The Avengers"]

for title in test_movies:
    print(f"\nRecommendations for: {title}")
    display(recommend_movies(title))


Recommendations for: Interstellar


,title,genres
0,Silent Running,Adventure Drama Science Fiction
1,2001: A Space Odyssey,Science Fiction Mystery Adventure
2,Lost in Space,Adventure Family Science Fiction
3,Mission to Mars,Science Fiction
4,Moonraker,Action Adventure Thriller Science Fiction



Recommendations for: Toy Story


,title,genres
0,Toy Story 3,Animation Family Comedy
1,Toy Story 2,Animation Comedy Family
2,Small Soldiers,Comedy Adventure Fantasy Science Fiction Action
3,The 40 Year Old Virgin,Comedy Romance
4,Child's Play 2,Drama Horror



Recommendations for: The Avengers


,title,genres
0,Avengers: Age of Ultron,Action Adventure Science Fiction
1,Fantastic Four,Action Adventure Science Fiction
2,Ant-Man,Science Fiction Action Adventure
3,Captain America: Civil War,Adventure Action Science Fiction
4,Iron Man 2,Adventure Action Science Fiction


In [25]:
def show_recommendations_with_scores(title, top_n=5):
    title = title.lower().strip()

    if title not in indices:
        print(f"Movie '{title}' not found.")
        return

    movie_idx = indices[title]
    scores = list(enumerate(cosine_sim[movie_idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [item for item in scores if item[0] != movie_idx][:top_n]

    result = movies.iloc[[item[0] for item in scores]][["title", "genres"]].copy()
    result["similarity_score"] = [round(item[1], 3) for item in scores]

    display(result.reset_index(drop=True))

show_recommendations_with_scores("Inception")

,title,genres,similarity_score
0,Duplex,Action Comedy Thriller,0.175
1,Cypher,Thriller Science Fiction Mystery,0.174
2,The Helix... Loaded,Action Comedy Science Fiction,0.161
3,The Count of Monte Cristo,Action Adventure Drama Thriller,0.152
4,Pitch Perfect 2,Comedy Music,0.147


In [26]:
genre_vectorizer = TfidfVectorizer(stop_words="english")

genre_matrix = genre_vectorizer.fit_transform(movies["genres"])

genre_similarity = cosine_similarity(genre_matrix)

print("Combined-content vocabulary:", len(tfidf.get_feature_names_out()))
print("Genre-only vocabulary:", len(genre_vectorizer.get_feature_names_out()))

Combined-content vocabulary: 5000
Genre-only vocabulary: 22


In [27]:
def recommend_by_genre(title, top_n=5):
    title = title.lower().strip()

    if title not in indices:
        print(f"Movie '{title}' not found.")
        return

    movie_idx = indices[title]
    scores = list(enumerate(genre_similarity[movie_idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [item for item in scores if item[0] != movie_idx][:top_n]

    result = movies.iloc[[item[0] for item in scores]][["title", "genres"]].copy()
    result["similarity_score"] = [round(item[1], 3) for item in scores]

    return result.reset_index(drop=True)

print("Combined-content recommendations:")
show_recommendations_with_scores("Inception")

print("\nGenre-only recommendations:")
display(recommend_by_genre("Inception"))

Combined-content recommendations:


,title,genres,similarity_score
0,Duplex,Action Comedy Thriller,0.175
1,Cypher,Thriller Science Fiction Mystery,0.174
2,The Helix... Loaded,Action Comedy Science Fiction,0.161
3,The Count of Monte Cristo,Action Adventure Drama Thriller,0.152
4,Pitch Perfect 2,Comedy Music,0.147



Genre-only recommendations:


,title,genres,similarity_score
0,Sky Captain and the World of Tomorrow,Mystery Action Thriller Science Fiction Adventure,1.000
1,Paycheck,Action Adventure Mystery Science Fiction Thriller,1.000
2,Congo,Action Adventure Drama Mystery Science Fiction...,0.973
3,Knowing,Action Adventure Drama Mystery Science Fiction...,0.973
4,Oblivion,Action Science Fiction Adventure Mystery,0.947


In [28]:
print("CONTENT-BASED FILTERING")
print("- Recommends movies with similar metadata.")
print("- Does not require user ratings or viewing history.")
print("- May recommend movies that are too similar.")

print("\nCOLLABORATIVE FILTERING")
print("- Uses preferences or ratings from multiple users.")
print("- Can discover movies with less obvious content similarities.")
print("- Requires user interaction data.")

CONTENT-BASED FILTERING
- Recommends movies with similar metadata.
- Does not require user ratings or viewing history.
- May recommend movies that are too similar.

COLLABORATIVE FILTERING
- Uses preferences or ratings from multiple users.
- Can discover movies with less obvious content similarities.
- Requires user interaction data.


In [29]:
def recommend_movies(title, top_n=5):
    title = title.lower().strip()

    if title not in indices:
        print(f"Movie '{title}' not found. Please try another title.")
        return

    movie_idx = indices[title]
    scores = list(enumerate(cosine_sim[movie_idx]))
    scores.sort(key=lambda x: x[1], reverse=True)

    scores = [
        (idx, score) for idx, score in scores
        if idx != movie_idx
    ][:top_n]

    result = movies.iloc[[idx for idx, score in scores]][
        ["title", "genres"]
    ].copy()

    result["similarity_score"] = [
        round(score, 3) for idx, score in scores
    ]

    return result.reset_index(drop=True)

In [30]:
movies_to_test = ["Inception", "The Dark Knight", "Toy Story"]

for title in movies_to_test:
    print(f"\nRecommendations for: {title}")
    display(recommend_movies(title))



Recommendations for: Inception


,title,genres,similarity_score
0,Duplex,Action Comedy Thriller,0.175
1,Cypher,Thriller Science Fiction Mystery,0.174
2,The Helix... Loaded,Action Comedy Science Fiction,0.161
3,The Count of Monte Cristo,Action Adventure Drama Thriller,0.152
4,Pitch Perfect 2,Comedy Music,0.147



Recommendations for: The Dark Knight


,title,genres,similarity_score
0,The Dark Knight Rises,Action Crime Drama Thriller,0.519
1,Batman Forever,Action Crime Fantasy,0.417
2,Batman Returns,Action Fantasy,0.408
3,Batman Begins,Action Crime Drama,0.407
4,Batman v Superman: Dawn of Justice,Action Adventure Fantasy,0.336



Recommendations for: Toy Story


,title,genres,similarity_score
0,Toy Story 3,Animation Family Comedy,0.520
1,Toy Story 2,Animation Comedy Family,0.490
2,Small Soldiers,Comedy Adventure Fantasy Science Fiction Action,0.307
3,The 40 Year Old Virgin,Comedy Romance,0.304
4,Child's Play 2,Drama Horror,0.194


In [31]:
print("MOVIE RECOMMENDATION PIPELINE")
print("1. Movie metadata loaded using Pandas")
print("2. Missing text values handled")
print("3. Genres and keywords extracted")
print("4. Movie metadata combined into text")
print("5. Text converted into TF-IDF vectors")
print("6. Cosine similarity calculated")
print("7. Top 5 similar movies recommended")

MOVIE RECOMMENDATION PIPELINE
1. Movie metadata loaded using Pandas
2. Missing text values handled
3. Genres and keywords extracted
4. Movie metadata combined into text
5. Text converted into TF-IDF vectors
6. Cosine similarity calculated
7. Top 5 similar movies recommended


In [32]:
project_summary = {
    "Dataset": "TMDB 5000 Movie Dataset",
    "Approach": "Content-Based Filtering",
    "Text Representation": "TF-IDF",
    "Similarity Metric": "Cosine Similarity",
    "Programming Language": "Python",
    "Main Library": "Scikit-learn",
    "Recommendations": "Top 5 similar movies"
}

for key, value in project_summary.items():
    print(f"{key}: {value}")

Dataset: TMDB 5000 Movie Dataset
Approach: Content-Based Filtering
Text Representation: TF-IDF
Similarity Metric: Cosine Similarity
Programming Language: Python
Main Library: Scikit-learn
Recommendations: Top 5 similar movies


In [33]:
interview_questions = [
    "1. What is content-based filtering?",
    "2. What is TF-IDF, and why did we use it?",
    "3. What is cosine similarity?",
    "4. Why do we combine genres, keywords, and overviews?",
    "5. What is one limitation of this recommendation system?"
]

for question in interview_questions:
    print(question)

1. What is content-based filtering?
2. What is TF-IDF, and why did we use it?
3. What is cosine similarity?
4. Why do we combine genres, keywords, and overviews?
5. What is one limitation of this recommendation system?
